# Market basket: luật mua kèm nào đáng đưa cho merchandiser

Merchandiser chuẩn bị catalogue mùa thu và hỏi: **"Nên bày gì cạnh gì, combo nào đáng làm?"**. Market basket analysis
tìm các luật "giỏ có A thì hay có B" từ hoá đơn, đo bằng ba con số: support (luật phổ biến tới đâu), confidence (trong
giỏ có A, bao nhiêu phần có B) và lift (confidence so với tỉ lệ nền của B).

Bài này tự tính ba con số bằng tay để hiểu chúng, rồi dùng `mlxtend` (FP-Growth + `association_rules`) cho toàn bộ
catalogue, và kiểm luật có **ổn định qua mùa** không. Dữ liệu: Online Retail II (UCI, CC BY 4.0), sheet
`Year 2010-2011`, dòng hợp lệ theo định nghĩa dùng chung. Một hoá đơn là một giỏ.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/12-product-customer-analytics/exercises/intermediate/02-market-basket-rules -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth

In [ ]:
# Viết sẵn, không cần sửa: đọc Online Retail II (cache parquet) và lọc dòng hợp lệ theo định nghĩa dùng chung
import tempfile
import uuid
from pathlib import Path

CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache trong thư mục tạm, không ghi vào data/raw


def read_sheet(sheet: str) -> pd.DataFrame:
    """Một sheet Online Retail II. Lần đầu đọc xlsx (30-90 giây) rồi cache parquet theo mtime và cỡ tệp."""
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    xlsx = root / "data" / "raw" / "online-retail-ii" / "online_retail_II.xlsx"
    if not xlsx.exists():
        raise FileNotFoundError("Chưa có dữ liệu. Chạy: uv run python data/download_dataset.py online-retail-ii")
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_II-{sheet.replace(' ', '_')}-{stat.st_mtime_ns}-{stat.st_size}.parquet"
    if cache.exists():
        try:
            return pd.read_parquet(cache)
        except (OSError, ValueError):  # cache hỏng: đọc lại
            pass
    tx = pd.read_excel(xlsx, sheet_name=sheet)
    tx = tx.assign(Invoice=tx["Invoice"].astype(str), StockCode=tx["StockCode"].astype(str),
                   Description=tx["Description"].astype("string"))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{uuid.uuid4().hex}.tmp")
    tx.to_parquet(tmp, index=False)
    tmp.replace(cache)
    return tx


def clean_sales(tx: pd.DataFrame) -> pd.DataFrame:
    """Bỏ hoá đơn huỷ (C...), Quantity <= 0, Price <= 0; thêm revenue = Quantity * Price."""
    s = tx.assign(Invoice=tx["Invoice"].astype(str))
    s = s[~s.Invoice.str.startswith("C") & (s.Quantity > 0) & (s.Price > 0)].copy()
    s["revenue"] = s.Quantity * s.Price
    return s

## 1. Ma trận giỏ và ba con số của một luật

`basket_matrix(lines) -> DataFrame`: `lines` có ít nhất cột `Invoice`, `StockCode`. Chỉ giữ **mã hàng hoá**: mã bắt
đầu bằng 5 chữ số (regex `^\d{5}`, giữ `85123A`, bỏ `POST`, `DOT`, `M`, `BANK CHARGES`...). Trả bảng `bool`, index
là `Invoice` (chuỗi), cột là `StockCode` (chuỗi), `True` khi giỏ có mã đó. Một mã nhiều dòng trong cùng hoá đơn vẫn
là một; hoá đơn chỉ có mã phí thì không thành giỏ.

`rule_stats(basket, a, b) -> dict`: ba khoá `support`, `confidence`, `lift` của luật `a ⇒ b`, tính trực tiếp từ các
cột `bool` (tỉ lệ giỏ có cả hai; chia cho tỉ lệ giỏ có `a`; chia tiếp cho tỉ lệ giỏ có `b`).

In [ ]:
def basket_matrix(lines: pd.DataFrame) -> pd.DataFrame:
    """Giỏ (Invoice) × mã hàng hoá (StockCode bắt đầu bằng 5 chữ số), giá trị bool."""
    # TODO
    raise NotImplementedError


def rule_stats(basket: pd.DataFrame, a: str, b: str) -> dict:
    """support, confidence, lift của luật a => b."""
    # TODO
    raise NotImplementedError

## 2. Khai thác luật cho cả catalogue và kiểm độ ổn định

`mine_rules(basket, min_support) -> DataFrame`: `fpgrowth(basket, min_support=..., use_colnames=True)` rồi
`association_rules(..., metric="lift", min_threshold=1.0)`. Chỉ giữ luật **một mã ⇒ một mã**. Trả bảng cột theo thứ tự
`antecedent`, `consequent` (chuỗi mã, không phải frozenset), `support`, `confidence`, `lift`; sắp theo `lift` giảm dần
rồi `antecedent` tăng dần; index `0..n-1`. Không có itemset hai mã nào đủ support thì trả bảng rỗng đúng 5 cột.

`stable_rules(r1, r2) -> DataFrame`: các luật (cùng `antecedent` và `consequent`) có mặt trong **cả hai** bảng, cột
số liệu mang hậu tố `_1`, `_2` (`support_1`, `lift_2`...).

In [ ]:
RULE_COLUMNS = ["antecedent", "consequent", "support", "confidence", "lift"]


def mine_rules(basket: pd.DataFrame, min_support: float) -> pd.DataFrame:
    """Luật một-một, lift >= 1, từ FP-Growth; cột RULE_COLUMNS; sắp lift giảm, antecedent tăng."""
    # TODO
    raise NotImplementedError


def stable_rules(r1: pd.DataFrame, r2: pd.DataFrame) -> pd.DataFrame:
    """Luật có trong cả hai bảng, hậu tố _1 và _2."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: giỏ thật, tên mã hàng, mã phổ biến nhất
sales = clean_sales(read_sheet("Year 2010-2011"))
basket = basket_matrix(sales)
goods = sales[sales.StockCode.str.match(r"^\d{5}")]
names = goods.groupby("StockCode").Description.agg(lambda d: d.dropna().mode().iloc[0].strip())
print(f"{basket.shape[0]:,} giỏ × {basket.shape[1]:,} mã hàng; trung vị {basket.sum(axis=1).median():.0f} mã mỗi giỏ")
top_items = basket.mean().nlargest(8)
print(top_items.rename(names).round(4).to_string())

In [ ]:
# explore: luật với min_support 2%, và min_support đổi thì số luật đổi ra sao
rules = mine_rules(basket, 0.02)
named = rules.assign(A=rules.antecedent.map(names), B=rules.consequent.map(names))
cols = ["A", "B", "support", "confidence", "lift"]
print(f"{len(rules)} luật một-một")
print("lift cao nhất:")
print(named[cols].head(6).round(3).to_string(index=False))
print("support cao nhất:")
print(named.sort_values("support", ascending=False)[cols].head(4).round(3).to_string(index=False))
print("lift thấp nhất:")
print(named[cols].tail(3).round(3).to_string(index=False))
for s in (0.03, 0.01):
    print(f"min_support {s:.0%}: {len(mine_rules(basket, s))} luật")
print("tự tính lại luật lift cao nhất:", {k: round(v, 4) for k, v in rule_stats(basket, rules.antecedent[0], rules.consequent[0]).items()})

In [ ]:
# explore: support × lift; kích thước điểm là confidence
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(rules.support * 100, rules.lift, s=rules.confidence * 80, alpha=0.5, color="#4c78a8")
for _, r in named.sort_values("support", ascending=False).head(3).iterrows():
    ax.annotate(f"{r.A[:22]} ⇒ {r.B[:22]}", (r.support * 100, r.lift), fontsize=7, xytext=(4, 4), textcoords="offset points")
for _, r in named.head(1).iterrows():
    ax.annotate(f"{r.A[:22]} ⇒ {r.B[:22]}", (r.support * 100, r.lift), fontsize=7, xytext=(4, -10), textcoords="offset points")
ax.set_xlabel("support (% giỏ có cả hai)")
ax.set_ylabel("lift")
ax.axhline(1, color="#888", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Lift cao nhất ở luật hiếm; luật phổ biến có lift vừa phải", loc="left")
plt.show()

In [ ]:
# explore: hai nửa năm, luật nào chỉ có ở mùa Giáng sinh
first = mine_rules(basket_matrix(sales[sales.InvoiceDate < "2011-07-01"]), 0.02)
second = mine_rules(basket_matrix(sales[sales.InvoiceDate >= "2011-07-01"]), 0.02)
both = stable_rules(first, second)
print(f"12/2010-06/2011: {len(first)} luật; 07-12/2011: {len(second)} luật; chung: {len(both)}")
only_second = second.merge(first, on=["antecedent", "consequent"], how="left", indicator=True)
only_second = only_second[only_second["_merge"] == "left_only"]
print("support cao nhất chỉ có ở nửa sau:")
print(only_second.assign(A=only_second.antecedent.map(names), B=only_second.consequent.map(names))
      .sort_values("support_x", ascending=False)[["A", "B", "support_x", "lift_x"]].head(5).round(3).to_string(index=False))

## Đọc luật, rồi làm trắc nghiệm

1. Luật lift cao nhất là gì? Đó có phải cơ hội **gợi ý** bán kèm, hay khách đã tự mua đủ bộ? Hành động hợp lý là gì?
2. Confidence của A ⇒ B và B ⇒ A khác nhau. Vì sao? Lift có khác không?
3. Luật giữa hai món bán chạy nhất có lift bao nhiêu? Confidence 18-19% nghĩa là gì khi một món có trong 11% mọi giỏ?
4. Hạ min_support từ 2% xuống 1% thì số luật tăng bao nhiêu lần? Bạn chọn ngưỡng thế nào cho merchandiser?
5. Bao nhiêu luật xuất hiện ở cả hai nửa năm? Một luật chỉ có ở nửa sau có nên dùng cho catalogue mùa xuân không?
6. Viết đề xuất một combo cho merchandiser, kèm cách đo nó có tăng doanh thu thật không.

Xong thì mở app, làm phần **trắc nghiệm** của bài.